In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [4]:
from pathlib import Path

DIR_OUTPUT = Path("./output")
DIR_DATA   = Path("./data")
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)
DIR_DATA.mkdir(parents=True, exist_ok=True)

do it frame by frame

-histogram equalization

-colormap_jet

-apply color balance adjustment

-logarithmic transformation

-power law transformation (gamma transformation)

In [5]:
#pre-compute lookup
x = np.arange(256, dtype=np.float32)

#log transform
log_lut = (255 / np.log(256)) * np.log(1+x)
# np.log(1 + x) stretches dark values and compresses bright ones.
# 255 / np.log(256) is the scaling constant, so the input 255 maps to exactly 255.

In [6]:
#power law
gamma = 1.5 #gamma > 1 to suppress bright backscatter
gamma_lut = 255 * (x/255) ** gamma

# Combine both into a single table: log first, then gamma
combined = np.interp(log_lut, x, gamma_lut)
LUT = np.clip(combined, 0, 255).astype(np.uint8) #.astype(np.uint8) converts to 8-bit, which cv2.LUT requires

In [7]:
def color_balance(img):
  f = img.astype(np.float32)
  b, g, r = cv2.split(f)
  mb, mg, mr = b.mean() + 1e-6, g.mean() + 1e-6, r.mean() + 1e-6
  gray = (mb+mg+mr) / 3
  b *= gray / mb
  g *= gray / mg
  r*= gray / mr
  return np.clip(cv2.merge([b, g, r]), 0, 255).astype(np.uint8)


def enhance(frame):
  gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
  eq = cv2.equalizeHist(gray)
  heatmap = cv2.applyColorMap(eq, cv2.COLORMAP_JET)
  balance = color_balance(heatmap)
  return cv2.LUT(balance, LUT)

In [10]:
video = cv2.VideoCapture(DIR_DATA / "0X1A0A263B22CCD966.mp4")
assert video.isOpened(), "Could not open the video. Check the path."

fps = video.get(cv2.CAP_PROP_FPS) or 30
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(str(DIR_OUTPUT / "enhanced.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w*2, h))
#from google.colab.patches import cv2_imshow
while True:
  ret, frame = video.read()
  if not ret: #end of video
    break
  processed = enhance(frame)
  side_by_side = np.hstack([frame, processed])
  out.write(side_by_side)
  #cv2_imshow(side_by_side)

video.release()
out.release()
print("Done")

Done
